# GEDI Footprint Extraction Workflow

Extracts GEDI Level 2A shots over an area of interest, filters to high-quality returns, and exports point and footprint (25 m buffer) GeoJSON files.

**Bounding box:** Set from a `.gdb` boundary file (interactive dialog) **or** entered manually below.

---
## Setup

Install required packages if you haven't already:

```bash
# in Google Colab:
#   !pip install earthaccess h5py numpy pandas geopandas shapely fiona -q
# in VS Code terminal:
#   python -m pip install earthaccess h5py numpy pandas geopandas shapely fiona
```

In [1]:
#------------ imports ----------------------------------------------------------
import earthaccess
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import datetime
import os
import tkinter as tk
from tkinter import filedialog, simpledialog
import fiona

c:\Users\davisk10\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Step 1: Set Output Folder and File Header

A dialog will open asking you to:
1. Select the folder where output files (CSV, GeoJSON) will be saved
2. Enter a short header used to name all output files (e.g. `bpines` → `bpines_gedi_shots.csv`)

In [2]:
data_folder = input("Enter output folder path: ").strip().strip('"')
file_header = input("Enter file name header (e.g. 'bpines'): ").strip().lower().replace(" ", "_")

csv_path            = os.path.join(data_folder, f'{file_header}_gedi_shots.csv')
geojson_path        = os.path.join(data_folder, f'{file_header}_gedi_shots.geojson')
buffer_geojson_path = os.path.join(data_folder, f'{file_header}_gedi_footprints.geojson')

print(f"\nOutput files will be:")
print(f"  {csv_path}")
print(f"  {geojson_path}")
print(f"  {buffer_geojson_path}")


Output files will be:
  C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.csv
  C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.geojson
  C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_footprints.geojson


---
## Step 2: Define Area of Interest Bounding Box

**Option A** — Load from a `.gdb` boundary file (recommended): run the cell below.

**Option B** — Enter coordinates manually: skip to the manual cell further below.

### Option A: Load boundary from .gdb

In [3]:
# Select the boundary GDB
gdb_path = input("Paste the path to your .gdb folder: ").strip().strip('"')

if not gdb_path:
    raise SystemExit("No .gdb folder path entered. Exiting.")

# List available layers
layers = fiona.listlayers(gdb_path)
print("\nLayers found in geodatabase:")
for i, layer in enumerate(layers):
    print(f"  {i + 1}: {layer}")

# Ask user to pick a layer
layer_index = int(input(f"\nEnter the number of the layer to use (1–{len(layers)}): ").strip())

if not layer_index:
    raise SystemExit("No layer selected. Exiting.")

layer_name = layers[layer_index - 1]
print(f"\nUsing layer: {layer_name}")

# Load the boundary layer and compute bounding box
boundary = gpd.read_file(gdb_path, layer=layer_name)
boundary_wgs84 = boundary.to_crs("EPSG:4326")

minx, miny, maxx, maxy = boundary_wgs84.total_bounds
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = minx, miny, maxx, maxy

print(f"\nBounding box derived from .gdb boundary:")
print(f"LAT_MIN, LAT_MAX = {LAT_MIN:.6f}, {LAT_MAX:.6f}")
print(f"LON_MIN, LON_MAX = {LON_MIN:.6f}, {LON_MAX:.6f}")


Layers found in geodatabase:
  1: Polygons_2
  2: treesandgardens_XYTableToPoint
  3: gedi_arb_points
  4: gedi_arb_points_Buffer
  5: blob_Buffer
  6: blob_Buffer2
  7: ZonalSt_gedi_ar1
  8: drone_percentile
  9: percentiles_drone_lidar
  10: percentiles_dron
  11: Line
  12: arb_blob
  13: redwood_trees
  14: GEDIFootprints25mDiameter_ExportTable
  15: GEDIFootprints25mDiameter_ExportFeatures
  16: gedi_arb_points_ExportFeatures
  17: gedi_arb_points_ExportF_Clip
  18: GEDI_Footprint_4_trees
  19: GEDI_Footprint_4_trees__ATTACH
  20: GEDI_Footprints_tallest_gt_trees1_trees_Merge
  21: GEDI_Footprints_tallest_gt_trees1_trees_Merge__ATTACH
  22: GEDI_Footprints_tallest_gt_trees1_trees_Merge1
  23: GEDI_Footprints_tallest_gt_trees1_trees_Merge1__ATTACH
  24: GEDI_Footprint_5_trees
  25: GEDI_Footprint_5_trees__ATTACH
  26: GEDI_Footprint_Merge
  27: GEDI_Footprint_Merge__ATTACH

Using layer: arb_blob

Bounding box derived from .gdb boundary:
LAT_MIN, LAT_MAX = 35.309664, 35.311200
LON_

### Option B: Insert coordinates manually

Only run this cell if you skipped Option A.

In [ ]:
# -------- Insert coordinates for area of interest -----------------------------
LAT_MIN, LAT_MAX = 35.309, 35.312
LON_MIN, LON_MAX = -120.664, -120.657

# Create a boundary polygon from the manual coordinates (needed for clipping in Step 9)
from shapely.geometry import box
boundary = gpd.GeoDataFrame(geometry=[box(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)], crs="EPSG:4326")

---
## Step 3: Login to NASA Earthdata

In [4]:
# ---------- Login to NASA Earthdata -------------------------------------------
# prompts for NASA Earthdata username and password
earthaccess.login()

---
## Step 4: Search for GEDI Granules

In [5]:
# ---------- Search for GEDI granules over area of interest --------------------
print('Searching for GEDI granules...')
results = earthaccess.search_data(
    short_name='GEDI02_A',
    version='002',
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
)
print(f'Found {len(results)} granules covering your arboretum')

Searching for GEDI granules...
Found 17 granules covering your arboretum


c:\Users\davisk10\miniconda3\Lib\site-packages\earthaccess\results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


---
## Step 5: Stream Granules and Extract Shots

Streams each granule directly — no download needed. (~20 min depending on granule count)

In [6]:
# --- Stream each granule and extract shots (~20 min) --------------------------
all_shots = []
for granule in results:
    filename = granule['meta']['native-id']
    print(f'\nProcessing: {filename}')

    try:
        # Stream the file directly — no download needed
        files = earthaccess.open([granule])
        h5file = h5py.File(files[0], 'r')
        beams = [k for k in h5file.keys() if k.startswith('BEAM')]

        for beam in beams:
            try:
                lat = h5file[beam]['lat_lowestmode'][:]
                lon = h5file[beam]['lon_lowestmode'][:]

                # Filter to arboretum bounding box
                mask = (
                    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
                    (lon >= LON_MIN) & (lon <= LON_MAX)
                )

                if mask.sum() == 0:
                    continue

                print(f'  {beam}: {mask.sum()} shots in arboretum')

                rh = h5file[beam]['rh'][mask]

                # Date from filename
                try:
                    year = int(filename[9:13])
                    doy  = int(filename[13:16])
                    date = datetime.datetime(year, 1, 1) + datetime.timedelta(doy - 1)
                    date_str = date.strftime('%Y-%m-%d')
                except:
                    date_str = 'unknown'

                shot_data = {
                    'filename':     filename,
                    'beam':         beam,
                    'date':         date_str,
                    'lat':          lat[mask],
                    'lon':          lon[mask],
                    'rh25':         rh[:, 25],
                    'rh50':         rh[:, 50],
                    'rh75':         rh[:, 75],
                    'rh75':         rh[:, 85],
                    'rh95':         rh[:, 95],  
                    'rh98':         rh[:, 98], #relative height at 98th percentile (top of forest canopy height in m)
                    'rh99':         rh[:, 99],
                    'rh100':        rh[:, 100],
                    'sensitivity':  h5file[beam]['sensitivity'][mask], #how well detected ground return through the canopy, ranges from 0 to 1
                                                                        #0.9 - 1.0 → excellent — laser punched through canopy cleanly and found ground
                    'quality_flag': h5file[beam]['quality_flag'][mask],
                    'degrade_flag': h5file[beam]['degrade_flag'][mask],
                    'shot_number':  h5file[beam]['shot_number'][mask].astype(str),
                }

                all_shots.append(pd.DataFrame(shot_data))

            except KeyError as e:
                print(f'  Skipping {beam} — missing field: {e}')
                continue

        h5file.close()

    except Exception as e:
        print(f'  Error: {e}')
        continue

c:\Users\davisk10\miniconda3\Lib\site-packages\earthaccess\store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)



Processing: GEDI02_A_2020112133647_O07692_02_T04072_02_003_01_V002


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 74.71it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:01<00:00,  1.88s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 6990.51it/s]


  BEAM0011: 4 shots in arboretum

Processing: GEDI02_A_2020200095731_O09054_03_T02271_02_003_01_V002


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 406.11it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 8439.24it/s]


...

QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 178.14it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 8256.50it/s]


  BEAM0011: 5 shots in arboretum

Processing: GEDI02_A_2025094173239_O35746_02_T06918_02_004_02_V002


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 1249.79it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:01<00:00,  1.45s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 7913.78it/s]


  BEAM0110: 4 shots in arboretum

Processing: GEDI02_A_2025179080244_O37058_02_T02802_02_004_02_V002


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 590.75it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 2385.84it/s]


---
## Step 6: Filter to High-Quality Shots and Save Outputs

In [7]:
# ------------- Combine all shots ----------------------------------------------
if len(all_shots) == 0:
    print('\nNo shots found in arboretum area')
else:
    combined = pd.concat(all_shots, ignore_index=True)
    print(f'\nTotal shots found: {len(combined)}')

    # Filter to high quality shots only
    quality = combined[
        (combined['quality_flag'] == 1) &
        (combined['degrade_flag'] == 0)
    ].copy()
    print(f'High quality shots: {len(quality)}')
    print(quality[['date', 'beam', 'lat', 'lon', 'rh98', 'sensitivity']].to_string())

    # Ensure shot_number stays as string (ArcGIS Pro compatibility)
    quality['shot_number'] = quality['shot_number'].astype(str)

    # Save CSV
    quality.to_csv(csv_path, index=False)
    print(f'\nCSV saved: {csv_path}')


Total shots found: 32
High quality shots: 18
          date      beam        lat         lon   rh98  sensitivity
0   2020-04-21  BEAM0011  35.309895 -120.663444   8.23     0.914987
1   2020-04-21  BEAM0011  35.310242 -120.662984  14.93     0.943995
2   2020-04-21  BEAM0011  35.310589 -120.662524   6.13     0.955509
3   2020-04-21  BEAM0011  35.310935 -120.662066   9.88     0.946373
12  2021-08-16  BEAM0011  35.310809 -120.662771  11.56     0.905081
16  2022-03-15  BEAM1000  35.309887 -120.662030   2.58     0.961871
17  2022-03-15  BEAM1000  35.310237 -120.661568   9.85     0.966369
18  2022-03-15  BEAM1000  35.310586 -120.661106   4.45     0.966245
19  2022-03-15  BEAM1000  35.310935 -120.660644   5.16     0.963093
23  2024-06-23  BEAM0011  35.309738 -120.663056   4.78     0.925517
24  2024-06-23  BEAM0011  35.310086 -120.662598   7.77     0.943286
25  2024-06-23  BEAM0011  35.310438 -120.662134  11.88     0.928673
26  2024-06-23  BEAM0011  35.310793 -120.661667   5.56     0.942738
27

---
## Step 7: Export Point GeoJSON

In [8]:
# ── Step 5: Build point GeoDataFrame ──────────────────────────────────────────
if len(quality) == 0:
    print('No quality shots to export. Exiting.')
else:
    geometry_pts = [Point(lon, lat) for lon, lat in zip(quality['lon'], quality['lat'])]
    gdf_points = gpd.GeoDataFrame(quality, geometry=geometry_pts, crs='EPSG:4326')

    # Save point GeoJSON
    gdf_points.to_file(geojson_path, driver='GeoJSON')
    print(f'Point GeoJSON saved: {geojson_path}')

Point GeoJSON saved: C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.geojson


---
## Step 8: Create 25 m Footprint Buffers and Export

**WHY WE REPROJECT:**  
GEDI points are stored in WGS84 (EPSG:4326), where coordinates are in degrees. Buffering in degrees does NOT give you metres — 0.0001° of longitude is ~9 m near the equator but changes with latitude.

Solution: reproject to a CRS measured in metres, buffer by exactly 12.5 m (radius), then reproject back to WGS84.

UTM Zone 10N (EPSG:32610) covers central California and is appropriate for this study area. Adjust the EPSG code if your ROI is in a different UTM zone.

**WHAT THE BUFFER DOES:**  
Each GEDI shot records a single lat/lon centre point, but the real laser footprint illuminates a ~25 m diameter circle on the ground. `buffer(12.5)` expands each point into a circle with radius 12.5 m, giving a 25 m diameter polygon that matches the physical footprint. When you later extract biomass, canopy height, or image pixel values WITHIN these polygons, you are sampling the same area the lidar saw.

In [9]:
# ── Step 6: Create 25 m footprint buffers ─────────────────────────────────────
UTM_CRS = 'EPSG:32610'   # UTM Zone 10N — change if your site is elsewhere

gdf_utm = gdf_points.to_crs(UTM_CRS)
gdf_utm['geometry'] = gdf_utm.geometry.buffer(12.5)    # 12.5 m radius → 25 m diameter
gdf_footprints = gdf_utm.to_crs('EPSG:4326')           # reproject back to WGS84

gdf_footprints.to_file(buffer_geojson_path, driver='GeoJSON')
print(f'Footprint GeoJSON saved (25 m buffers): {buffer_geojson_path}')

print('\nDone! Load the GeoJSON files into QGIS or ArcGIS Pro.')
print('Outputs:')
print(f'  Points     → {geojson_path}')
print(f'  Footprints → {buffer_geojson_path}')
print(f'  CSV        → {csv_path}')

Footprint GeoJSON saved (25 m buffers): C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_footprints.geojson

Done! Load the GeoJSON files into QGIS or ArcGIS Pro.
Outputs:
  Points     → C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.geojson
  Footprints → C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_footprints.geojson
  CSV        → C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.csv


---
## Diagnostic: Why Were Shots Rejected?

Re-streams all granules without quality filtering and prints a breakdown of rejection reasons. Run this if you want to understand why shots were excluded.

In [ ]:
# ------ Diagnostic: why were shots rejected? ----------------------------------

all_shots_unfiltered = []

for granule in results:
    filename = granule['meta']['native-id']

    try:
        files = earthaccess.open([granule])
        h5file = h5py.File(files[0], 'r')
        beams = [k for k in h5file.keys() if k.startswith('BEAM')]

        for beam in beams:
            try:
                lat = h5file[beam]['lat_lowestmode'][:]
                lon = h5file[beam]['lon_lowestmode'][:]

                mask = (
                    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
                    (lon >= LON_MIN) & (lon <= LON_MAX)
                )

                if mask.sum() == 0:
                    continue

                quality_flag = h5file[beam]['quality_flag'][mask]
                degrade_flag = h5file[beam]['degrade_flag'][mask]
                sensitivity  = h5file[beam]['sensitivity'][mask]

                # Date from filename
                try:
                    year = int(filename[9:13])
                    doy  = int(filename[13:16])
                    date = datetime.datetime(year, 1, 1) + datetime.timedelta(doy - 1)
                    date_str = date.strftime('%Y-%m-%d')
                except:
                    date_str = 'unknown'

                for i in range(mask.sum()):
                    all_shots_unfiltered.append({
                        'filename':     filename,
                        'beam':         beam,
                        'date':         date_str,
                        'lat':          lat[mask][i],
                        'lon':          lon[mask][i],
                        'quality_flag': quality_flag[i],
                        'degrade_flag': degrade_flag[i],
                        'sensitivity':  sensitivity[i],
                        'rejected':     quality_flag[i] != 1 or degrade_flag[i] != 0
                    })

            except KeyError as e:
                continue

        h5file.close()

    except Exception as e:
        print(f'Error: {e}')
        continue

# ── Summary ───────────────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_shots_unfiltered)

print(f'Total shots in bounding box: {len(df_all)}')
print(f'Accepted (quality=1, degrade=0): {len(df_all[~df_all["rejected"]])}')
print(f'Rejected: {len(df_all[df_all["rejected"]])}')
print()
print('Rejection breakdown:')
print(f'  quality_flag = 0: {len(df_all[df_all["quality_flag"] == 0])}')
print(f'  degrade_flag != 0: {len(df_all[df_all["degrade_flag"] != 0])}')
print()
print('Rejected shots by date and beam:')
rejected = df_all[df_all['rejected']]
print(rejected[['date', 'beam', 'lat', 'lon', 'quality_flag', 'degrade_flag', 'sensitivity']].to_string())

---
## Step 9: Clip GEDI Footprints to Previously Entered Study Area Bounday 

In [13]:
## Step 9: Clip Footprints to Study Area Boundary

# Reproject boundary to WGS84 to match footprints
boundary_wgs84 = boundary.to_crs("EPSG:4326")
boundary_union = boundary_wgs84.union_all()  # single geometry for comparison

# Keep only footprints completely within the boundary
gdf_footprints_clipped = gdf_footprints[gdf_footprints.geometry.within(boundary_union)].copy()

# Match points to surviving footprints using shot_number
gdf_points_clipped = gdf_points[gdf_points['shot_number'].isin(gdf_footprints_clipped['shot_number'])].copy()

print(f"Footprints before filtering:      {len(gdf_footprints)}")
print(f"Footprints fully within boundary: {len(gdf_footprints_clipped)}")
print(f"Points matched to footprints:     {len(gdf_points_clipped)}")

# Save outputs
clipped_footprints_path = os.path.join(data_folder, f'{file_header}_gedi_footprints_clipped.geojson')
clipped_points_path     = os.path.join(data_folder, f'{file_header}_gedi_shots_clipped.geojson')

gdf_footprints_clipped.to_file(clipped_footprints_path, driver='GeoJSON')
gdf_points_clipped.to_file(clipped_points_path, driver='GeoJSON')

print(f"\nClipped footprints saved: {clipped_footprints_path}")
print(f"Clipped points saved:     {clipped_points_path}")

Footprints before filtering:      18
Footprints fully within boundary: 6
Points matched to footprints:     6

Clipped footprints saved: C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_footprints_clipped.geojson
Clipped points saved:     C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots_clipped.geojson


At this point in the code, (with the hope that i am somehow able to connect the R code for creating the CHM for the study area before), I should have:
    1. the CHM for the study area
    2. the GEDI footprint locations 
    3. the GEDI height metrics for each footprint
    4. the buffered 12.5m radius rings for each footprint

What I will need to do next in this code is the following: 
1. incoroporate the Monte Carlo algorithm (I think it is better than the UAV paper method because the randomization is better i think?)


Monte Carlo Simulation-Based Correction of Geolocation Errors in GEDI Footprint Positions
==========================================================================================
Based on:
  - 'Simulation-Based Correction of Geolocation Errors in GEDI Footprint Positions
     Using Monte Carlo Approach'
  - 'The impact of geolocation uncertainty on GEDI tropical forest canopy height
     estimation and change monitoring'

Method:
  Each GEDI footprint location is shifted with randomly generated position errors
  modelled using the GEDI geolocation uncertainty (Dubayah et al., 2020a):

      x*_i = x + s_i * cos(theta_i)
      y*_i = y + s_i * sin(theta_i)

  where:
    (x*_i, y*_i) = shifted GEDI footprint centre coordinate
    (x, y)       = GEDI product reported footprint centre coordinate
    s_i          ~ N(mu=0 m, sigma=10 m)  [geolocation uncertainty]
    theta_i      ~ Uniform(0, 2*pi)       [random direction]
    n            = 300 simulations per footprint